# Marginal input comparison

Compare decoders on independent BPSK/AWGN samples with a uniformly sampled SNR in dB. Every decoder receives the **same observations**, with its own iteration count. Edit the configuration, run the experiment, then inspect timing and conditional SER distributions.

SNR means **Es/N0**, with Es=1. For real noise, variance is $\sigma^2=1/(2\cdot10^{\mathrm{SNR}_{dB}/10})$. This differs by a factor of two from signal power divided by noise variance. No code-rate or Eb/N0 conversion is applied.

In [8]:
import importlib
import sys
from functools import partial
from pathlib import Path

from IPython.display import display

notebooks_dir = (
    Path.cwd() if (Path.cwd() / "_shared").is_dir() else Path.cwd() / "notebooks"
)
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from _shared.application.marginal_comparison import run_experiment  # noqa: E402
from _shared.core.decoders import avail_softmajvote  # noqa: E402
from _shared.core.decoders._adt import NormalizedMinSum, SqrtSign, Tanh  # noqa: E402
from _shared.core.lfsr import polynomial_terms  # noqa: E402

SNR_DB_RANGE = (-3, 3)
P = polynomial_terms((0, 2, 3, 5, 16))  # x^16 + x^5 + x^3 + x^2 + 1
K = P[-1]  # 16; change P to change K. Primitivity is trusted.
N = 32
BATCH_SIZE = 1024
BATCHES = 1024
WORKERS = None  # All CPU cores available to joblib; one process per worker.
SEED = 0

avail_softmajvote = importlib.reload(avail_softmajvote)

DECODERS = {
    "avail-softmajvote(tanh, T=5)": partial(
        avail_softmajvote.decode, softxor=Tanh(), iterations=5
    ),
    "avail-softmajvote(sqrt-sign, T=3)": partial(
        avail_softmajvote.decode, softxor=SqrtSign(), iterations=3
    ),
    "avail-softmajvote(min-sum(0.8), T=5)": partial(
        avail_softmajvote.decode,
        softxor=NormalizedMinSum(coefficient=0.8),
        iterations=5,
    ),
}

## Compile and warm up Numba

Run this cell before the experiment. A disposable batch with the configured `BATCH_SIZE`, `N` and `P` exercises the encoder and every selected decoder in final-only mode. This triggers compilation (or loads the on-disk Numba cache) separately from the experiment. The elapsed times below include warmup and are **not decoding benchmarks**.

Warmup uses an independent random generator and does not consume experiment seeds. Worker initialization still loads and warms compiled code in each process before timed decoding. Rerun this cell after changing decoder implementations or configurations.

In [9]:
from time import perf_counter

import numpy as np
from _shared.core.channel import sample_batch
from tqdm.auto import tqdm

warmup_started = perf_counter()
warmup_llr, _, _ = sample_batch(
    np.random.default_rng(0), BATCH_SIZE, N, P, SNR_DB_RANGE
)
for name, decoder in tqdm(DECODERS.items(), desc="Numba warmup", unit="decoder"):
    started = perf_counter()
    decoder(warmup_llr.copy(), terms=P, return_history=False)
    print(f"{name}: {perf_counter() - started:.3f} s (compile/cache + warmup)")
del warmup_llr
print(f"Warmup complete in {perf_counter() - warmup_started:.3f} s")

Numba warmup:   0%|          | 0/3 [00:00<?, ?decoder/s]

avail-softmajvote(tanh, T=5): 0.060 s (compile/cache + warmup)
avail-softmajvote(sqrt-sign, T=3): 0.004 s (compile/cache + warmup)
avail-softmajvote(min-sum(0.8), T=5): 0.001 s (compile/cache + warmup)
Warmup complete in 0.073 s


## Generate and decode batches

For each sample:

1. Draw independent uniform binary information bits x and uniform SNR in `SNR_DB_RANGE`.
2. Encode with the sparse LFSR, then map coded bits to $s=1-2y$, so Es=1.
3. Generate $r=s+\mathcal N(0,\sigma^2)$ and feed $L=2r/\sigma^2$ to the decoder.

One process-pool task generates one batch, runs every decoder, and returns per-decoder batch decode time, per-sample sign error rates, and each sample's SNR. Sample seeds do not depend on scheduling. `tqdm` counts **completed batches**.

The public decoder contract is `decode(initial_batch, terms=..., iterations=..., return_history=False, ...)`. Input and final output have shape `(B, N)`; optional histories have shape `(B, T+1, N)`. Final-only mode uses rolling buffers. Decoders never receive truth.

SER here means **sign error rate across all N coded bits**, not symbol/block error rate or information-bit BER. Exact-zero LLRs count as half an error. Worker warmup excludes JIT compilation from reported decode time; generation, copies, statistics and interprocess transfer are also excluded. Timing includes decoder validation and output allocation, and reflects concurrent CPU load.

In [10]:
experiment = run_experiment(
    DECODERS,
    terms=P,
    n=N,
    snr_db_range=SNR_DB_RANGE,
    batch_size=BATCH_SIZE,
    batches=BATCHES,
    workers=WORKERS,
    seed=SEED,
)
print(f"Completed {BATCH_SIZE * BATCHES:,} paired samples per decoder.")

Batches:   0%|          | 0/1024 [00:00<?, ?batch/s]

Completed 1,048,576 paired samples per decoder.


## Mean decoding time

Total measured batch decoding time divided by the number of samples, in nanoseconds per sample. This is amortized batch throughput timing, not isolated single-sample latency.

In [11]:
import importlib

import _shared.presentation.marginal_plots as marginal_plots

marginal_plots = importlib.reload(marginal_plots)
display(marginal_plots.timing_table(experiment))

Decoder,Mean decode time (ns/sample)
"avail-softmajvote(tanh, T=5)","77,754.8"
"avail-softmajvote(sqrt-sign, T=3)","10,900.5"
"avail-softmajvote(min-sum(0.8), T=5)","2,970.0"


## Conditional SER distributions

The field is split into `SNR_BINS` columns and `SER_BINS` rows covering SER [0, 1]. Each cell counts observations. Counts are divided by the total in that SNR column, so every populated column sums to one independently for each decoder. Empty columns remain gaps.

Cell opacity is linear in **P(SER cell | SNR column)**, using the same fixed [0, 1] scale for all decoders. No normal-distribution assumption, quantile bands, smoothing, or density interpolation is used. Hover reports cell probability and count. Overlaid decoder colors blend; select a single decoder to inspect its distribution alone.

The line remains the mean of the original SER samples in each SNR column. Changing bin counts only redraws completed results; decoding does not run again.


In [12]:
import _shared.application.marginal_comparison as marginal_comparison
import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

marginal_comparison = importlib.reload(marginal_comparison)
plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
marginal_plots = importlib.reload(marginal_plots)
SNR_BINS = 30
SER_BINS = N
marginal_view = marginal_plots.distributions(
    experiment,
    bins=SNR_BINS,
    ser_bins=SER_BINS,
    previous=globals().get("marginal_view"),
)
display(marginal_view)

ComparisonView(children=(FigureWidget({
    'data': [{'colorscale': [[0, 'rgba(99,110,250,0)'], [1, 'rgba(99,1…